# Contrastive Analyst — Qwen3.5-9B

같은 task에서 나온 (실패, 성공) trajectory 쌍에 `prompts/contrastive_analyst.md`를 적용해 결과를 확인합니다.

- 입력: `data/pairs/<benchmark>.jsonl` (로컬에서 `scripts/build_pairs.py`로 생성해 레포에 포함)
- 모델: `Qwen/Qwen3.5-9B` (vLLM OpenAI 호환 서버, thinking off, JSON schema 제약 디코딩)
- 출력: `outputs/<TAG>/` → Drive에 복사

순서: 1 설정 → 2 Drive → 3 레포 → 4 설치(**설치 후 런타임 재시작**) → 5 서버 → 6 스모크 테스트 → 7 본 실행 → 8 결과

In [ ]:
# 1. 설정
REPO_URL = "https://github.com/<YOUR_ID>/<YOUR_REPO>.git"   # ← 본인 레포로 바꾸세요
BRANCH = "main"
REPO_DIR = "/content/contrastive-skill"
MODEL_ID = "Qwen/Qwen3.5-9B"
PORT = 8000
TAG = "analyst_v1"                       # 출력 폴더 이름 (프롬프트를 바꾸면 TAG도 바꾸세요)
BENCHMARKS = ["officeqa", "spreadsheetbench", "livemath", "docvqa", "searchqa"]
LIMIT = 30                               # 벤치마크당 최대 쌍 수 (0 = 전체; natural 쌍을 우선 포함)
DRIVE_OUT = f"/content/drive/MyDrive/contrastive_skill/{TAG}"

In [ ]:
# 2. Google Drive (결과 보관용)
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_OUT, exist_ok=True)
print("results ->", DRIVE_OUT)

In [ ]:
# 3. 레포 받기 (이미 있으면 최신으로 갱신). private 레포면 Colab Secrets에 GITHUB_TOKEN을 등록하세요.
import os, subprocess
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# 4. 설치 — 끝나면 [런타임 > 세션 다시 시작] 후 1, 3번 셀을 다시 실행하고 5번으로 가세요.
# torchaudio는 vllm이 끌어오는 torch와 CUDA 빌드가 달라 import 체인을 깨뜨리므로 제거합니다 (SKILLOPT-COLAB-SETUP 11.5).
!pip install -q -U vllm openai json_repair
!pip uninstall -y -q torchaudio
!python -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# 5. vLLM 서버 기동
import os, subprocess, time, requests, torch
LOG_PATH = "/content/vllm_server.log"
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
MAX_MODEL_LEN = 65536 if gpu_gb >= 35 else 32768   # OfficeQA/SpreadsheetBench 일부 쌍은 4~5만 토큰
print(torch.cuda.get_device_name(0), f"{gpu_gb:.0f}GB", "MAX_MODEL_LEN =", MAX_MODEL_LEN)

env = dict(os.environ, VLLM_USE_FLASHINFER_SAMPLER="0")   # Blackwell 샘플러 함정 회피 (11.7)
cmd = ["python", "-m", "vllm.entrypoints.openai.api_server",
       "--model", MODEL_ID, "--served-model-name", MODEL_ID, "--port", str(PORT),
       "--max-model-len", str(MAX_MODEL_LEN), "--gpu-memory-utilization", "0.90",
       "--limit-mm-per-prompt", '{"image": 1}', "--trust-remote-code"]
proc = subprocess.Popen(cmd, stdout=open(LOG_PATH, "w"), stderr=subprocess.STDOUT, env=env, start_new_session=True)

for _ in range(180):
    if proc.poll() is not None:
        break
    try:
        if requests.get(f"http://localhost:{PORT}/v1/models", timeout=2).ok:
            print("server ready"); break
    except Exception:
        pass
    time.sleep(10)
else:
    print("timeout")
if proc.poll() is not None or not requests.get(f"http://localhost:{PORT}/v1/models", timeout=2).ok:
    txt = open(LOG_PATH).read().splitlines()
    eng = [l for l in txt if "EngineCore" in l] or txt
    print("
".join(eng[-80:]))

In [ ]:
# 6. 스모크 테스트: 벤치마크당 2쌍, 별도 TAG로 저장
!python scripts/run_analyst.py --base-url http://localhost:{PORT}/v1 --model {MODEL_ID} --tag smoke --limit 2 --benchmarks {" ".join(BENCHMARKS)}

In [ ]:
# 7. 본 실행 (중단돼도 같은 TAG로 다시 돌리면 끝난 쌍은 건너뜁니다)
!python scripts/run_analyst.py --base-url http://localhost:{PORT}/v1 --model {MODEL_ID} --tag {TAG} --limit {LIMIT} --benchmarks {" ".join(BENCHMARKS)}

In [ ]:
# 8. 결과 요약 + Drive 복사
import shutil
from IPython.display import Markdown, display
out = f"{REPO_DIR}/outputs/{TAG}"
display(Markdown(open(f"{out}/summary.md", encoding="utf-8").read()))
shutil.copytree(out, DRIVE_OUT, dirs_exist_ok=True)
shutil.copy(LOG_PATH, DRIVE_OUT)
print("copied to", DRIVE_OUT)

In [ ]:
# 9. 개별 결과 보기
import json
BENCH, N = "officeqa", 3
rows = [json.loads(l) for l in open(f"{REPO_DIR}/outputs/{TAG}/{BENCH}.jsonl", encoding="utf-8")]
for r in rows[:N]:
    print("=" * 100, "
", r["pair_id"], r["source"], "json_ok:", r["json_ok"], r.get("error", ""))
    print(json.dumps(r["parsed"], indent=2, ensure_ascii=False) if r["parsed"] else r["raw"])